# Result and provenance explorer

This notebook inspects the completed `RES-BB-SYN-006` misspecification and negative-control suite without rerunning it. It validates plan, code, environment, seed, status, and artifact identity; then creates cell-level, paired-method, approximation, and resource plots.

The suite maps predeclared failure regimes. It is **not** a universal robustness claim. Failed, null, reversed, nonconverged, and timed-out outcomes remain visible and are never imputed.

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bayesbreak

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
RESULT_DIR = ROOT / "results" / "phase6" / "RES-BB-SYN-006"
RESULT_PATH = RESULT_DIR / "results.json"
PLAN_PATH = ROOT / "provenance" / "epr-bb-015-plan.json"
OUTPUT_DIR = ROOT / "results" / "notebook_verification" / "result_explorer"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
plan = json.loads(PLAN_PATH.read_text(encoding="utf-8"))
artifact_sha256 = hashlib.sha256(RESULT_PATH.read_bytes()).hexdigest()
plan_sha256 = hashlib.sha256(PLAN_PATH.read_bytes()).hexdigest()
print({"python": platform.python_version(), "bayesbreak": bayesbreak.__version__, "result_bytes": RESULT_PATH.stat().st_size, "result_sha256": artifact_sha256})

## 1. Verify artifact identity and execution authorization

The full result must identify the approved plan, exact clean code commit, environment hash, deterministic seed schedule, and certified pilot used as the authorization basis. These checks validate lineage; they do not validate the scientific conclusions by themselves.

In [ ]:
pilot_path = ROOT / plan["full_execution_approval"]["basis_pilot"]
pilot_sha256 = hashlib.sha256(pilot_path.read_bytes()).hexdigest()
identity_checks = [
    ("result id", result["result_id"] == "RES-BB-SYN-006", result["result_id"]),
    ("protocol id", result["protocol_id"] == "EPR-BB-015", result["protocol_id"]),
    ("full mode", result["mode"] == "full", result["mode"]),
    ("approved flag", result["full_execution_approved"] is True, result["full_execution_approved"]),
    ("plan hash", result["plan_sha256"] == plan_sha256, result["plan_sha256"]),
    ("pilot approval hash", pilot_sha256 == plan["full_execution_approval"]["basis_pilot_sha256"], pilot_sha256),
    ("clean relevant paths", result["code"]["relevant_paths_clean"] is True, result["code"]["relevant_paths_clean"]),
    ("record count", len(result["records"]) == 400, len(result["records"])),
    ("environment hash length", len(result["environment"]["sha256"]) == 64, result["environment"]["sha256"]),
]
identity_frame = pd.DataFrame(identity_checks, columns=["check", "passed", "observed"])
display(identity_frame)
print({"commit": result["code"]["commit"], "plan_sha256": result["plan_sha256"], "pilot_sha256": pilot_sha256})
assert identity_frame.passed.all()

## 2. Audit seeds, hashes, and retained statuses

Every generated dataset is the uncertainty unit. This section reconstructs the declared seed schedule, verifies all input-identity hashes, and counts top-level and nested method statuses without filtering timed-out approximation runs.

In [ ]:
cell_order = result["cell_ids"]
expected_seeds = {
    (cell_id, repetition): result["seed_base"] + 10_000 * cell_index + repetition
    for cell_index, cell_id in enumerate(cell_order)
    for repetition in range(result["repetitions_per_cell"])
}
observed_pairs = []
for cell_id in cell_order:
    cell_records = [record for record in result["records"] if record["cell"] == cell_id]
    for repetition, record in enumerate(cell_records):
        observed_pairs.append((cell_id, repetition, record["seed"], expected_seeds[(cell_id, repetition)]))
seed_frame = pd.DataFrame(observed_pairs, columns=["cell", "repetition", "observed_seed", "expected_seed"])

hash_names = ["data_hash", "truth_hash", "effective_config_hash"]
hash_audit = pd.DataFrame(
    [
        {"cell": record["cell"], "seed": record["seed"], **{name: isinstance(record.get(name), str) and len(record[name]) == 64 for name in hash_names}}
        for record in result["records"]
    ]
)
top_status_frame = pd.DataFrame(result["records"])[["cell", "seed", "status", "wall_seconds"]]
logistic_records = [record for record in result["records"] if record["cell"] == "logistic-approximation-failure"]
method_status_frame = pd.DataFrame(
    [{"seed": record["seed"], "method": method, "status": method_record["status"]} for record in logistic_records for method, method_record in record["methods"].items()]
)

display(top_status_frame.groupby(["cell", "status"]).size().rename("count").reset_index())
display(method_status_frame.groupby(["method", "status"]).size().rename("count").reset_index())
assert (seed_frame.observed_seed == seed_frame.expected_seed).all()
assert hash_audit[hash_names].all().all()
assert len(top_status_frame) == 400

## 3. Inspect cell-level estimates and uncertainty

Continuous summaries use dataset-level $t$ intervals; binary rates use Wilson score intervals. Plotting functions read the recorded intervals directly rather than recomputing or dropping unfavorable outcomes.

In [ ]:
cell_summaries = result["summary"]["cells"]
standard_cells = cell_order[:6]
metric_rows = []
for cell_id in standard_cells:
    summary = cell_summaries[cell_id]
    for metric in ["boundary_f1", "k_error", "missed_change_rate", "complete_boundary_recovery_rate", "map_saturation_rate"]:
        interval = summary.get(metric)
        if interval is not None:
            metric_rows.append({"cell": cell_id, "metric": metric, **interval})
metric_frame = pd.DataFrame(metric_rows)
display(metric_frame.pivot(index="cell", columns="metric", values="mean"))

f1_frame = metric_frame[metric_frame.metric == "boundary_f1"].copy()
rate_selection = pd.DataFrame(
    [
        {"cell": "null-gaussian", "metric": "false positive dataset", **cell_summaries["null-gaussian"]["false_positive_dataset_rate"]},
        {"cell": "zero-inflated-poisson", "metric": "MAP at k_max", **cell_summaries["zero-inflated-poisson"]["map_saturation_rate"]},
        {"cell": "dense-gaussian", "metric": "MAP at k_max", **cell_summaries["dense-gaussian"]["map_saturation_rate"]},
        {"cell": "short-segment-gaussian", "metric": "exact recovery", **cell_summaries["short-segment-gaussian"]["complete_boundary_recovery_rate"]},
        {"cell": "prior-conflict-gaussian", "metric": "exact recovery", **cell_summaries["prior-conflict-gaussian"]["complete_boundary_recovery_rate"]},
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
positions = np.arange(len(f1_frame))
f1_errors = np.vstack([np.maximum(0.0, f1_frame["mean"] - f1_frame.ci95_lower), np.maximum(0.0, f1_frame.ci95_upper - f1_frame["mean"])])
axes[0].errorbar(positions, f1_frame["mean"], yerr=f1_errors, fmt="o", capsize=4, color="#00798C")
axes[0].set(title="Boundary F1 by predeclared cell", ylabel="mean F1 and 95% t interval", xticks=positions, xticklabels=f1_frame.cell, ylim=(-0.05, 1.05))
axes[0].tick_params(axis="x", rotation=35)
positions = np.arange(len(rate_selection))
rate_errors = np.vstack([np.maximum(0.0, rate_selection["mean"] - rate_selection.ci95_lower), np.maximum(0.0, rate_selection.ci95_upper - rate_selection["mean"])])
axes[1].errorbar(positions, rate_selection["mean"], yerr=rate_errors, fmt="o", capsize=4, color="#D1495B")
axes[1].set(title="Selected failure-mode rates", ylabel="rate and 95% Wilson interval", xticks=positions, xticklabels=[f"{cell}\n{metric}" for cell, metric in zip(rate_selection.cell, rate_selection.metric, strict=True)], ylim=(-0.05, 1.05))
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cell_failure_map.png", bbox_inches="tight")
plt.show()

## 4. Analyze the paired shared-boundary contrast

For every generated dataset, shared and independent fits are scored against the same per-subject truth before averaging. The paired difference is `independent mean F1 − shared mean subject F1`; positive values favor the independent fit on this declared heterogeneous regime only.

In [ ]:
from scipy.stats import t as student_t

shared_records = [record for record in result["records"] if record["cell"] == "shared-boundary-heterogeneity"]
for record in shared_records:
    assert all(shared_subject["truth_boundaries"] == independent_subject["truth_boundaries"] for shared_subject, independent_subject in zip(record["shared_subject_metrics"], record["independent"], strict=True))
paired_frame = pd.DataFrame(
    [
        {
            "seed": record["seed"],
            "shared_mean_subject_f1": record["shared_mean_subject_f1"],
            "independent_mean_f1": record["independent_mean_f1"],
            "difference": record["independent_mean_f1"] - record["shared_mean_subject_f1"],
            "subject_deviation_selected": record["subject_specific_boundary_60_selected_as_shared"],
        }
        for record in shared_records
    ]
)
paired_mean = paired_frame.difference.mean()
paired_se = paired_frame.difference.std(ddof=1) / np.sqrt(len(paired_frame))
paired_critical = student_t.ppf(0.975, df=len(paired_frame) - 1)
paired_interval = (paired_mean - paired_critical * paired_se, paired_mean + paired_critical * paired_se)
print({"paired_mean_difference": paired_mean, "ci95": paired_interval, "deviation_selected_rate": paired_frame.subject_deviation_selected.mean()})

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
axes[0].scatter(paired_frame.shared_mean_subject_f1, paired_frame.independent_mean_f1, color="#00798C", alpha=0.7)
axes[0].plot([0, 1], [0, 1], color="#D1495B", linestyle="--")
axes[0].set(title="Paired dataset-level F1", xlabel="shared mean subject F1", ylabel="independent mean F1", xlim=(0, 1), ylim=(0, 1))
axes[1].hist(paired_frame.difference, bins=12, color="#EDAE49", edgecolor="white")
axes[1].axvline(0, color="#D1495B", linestyle="--")
axes[1].axvline(paired_mean, color="#00798C", linewidth=2, label=f"mean={paired_mean:.3f}")
axes[1].set(title="Paired F1 difference", xlabel="independent − shared", ylabel="datasets")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "shared_paired_contrast.png", bbox_inches="tight")
plt.show()

## 5. Inspect logistic approximation outcomes

Timeout is a scientific outcome. Diagnostics are summarized only for methods that actually executed; missing EP error metrics remain missing rather than being replaced with zero or copied from another approximation.

In [ ]:
approximation_rows = []
for record in logistic_records:
    for method, method_record in record["methods"].items():
        row = {"seed": record["seed"], "method": method, "status": method_record["status"], "wall_seconds": method_record.get("wall_seconds", np.nan), "fit_wall_seconds": method_record.get("fit_wall_seconds", np.nan), "max_block_error": np.nan, "empirical_tv": np.nan, "truth_f1": np.nan, "map_jaccard": np.nan}
        if method_record["status"] == "executed":
            extra = method_record["diagnostics"]["extra"]
            row.update({"max_block_error": extra["block_error_max"], "empirical_tv": extra["pk_tv_empirical"], "truth_f1": method_record["truth_metrics"]["f1"], "map_jaccard": extra["map_path_jaccard"]})
        approximation_rows.append(row)
approximation_frame = pd.DataFrame(approximation_rows)
approximation_summary = approximation_frame.groupby("method", as_index=False).agg(execution_rate=("status", lambda status: np.mean(status == "executed")), timeout_rate=("status", lambda status: np.mean(status == "timed-out")), max_block_error_mean=("max_block_error", "mean"), empirical_tv_mean=("empirical_tv", "mean"), truth_f1_mean=("truth_f1", "mean"), wall_seconds_mean=("wall_seconds", "mean"))
display(approximation_summary)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
positions = np.arange(len(approximation_summary))
axes[0].bar(positions - 0.18, approximation_summary.execution_rate, width=0.36, label="executed", color="#2A9D8F")
axes[0].bar(positions + 0.18, approximation_summary.timeout_rate, width=0.36, label="timed out", color="#D1495B")
axes[0].set(title="Approximation outcome rates", ylabel="dataset rate", xticks=positions, xticklabels=approximation_summary.method, ylim=(0, 1.05))
axes[0].legend()
executed_only = approximation_frame[approximation_frame.status == "executed"]
for method, group in executed_only.groupby("method"):
    axes[1].scatter(group.max_block_error, group.empirical_tv, label=method, alpha=0.65)
axes[1].set(title="Recorded approximation error", xlabel="maximum reachable-block error", ylabel="empirical posterior TV")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "logistic_approximations.png", bbox_inches="tight")
plt.show()
assert approximation_frame.loc[approximation_frame.status != "executed", ["max_block_error", "empirical_tv", "truth_f1"]].isna().all().all()

## 6. Explore individual records and resource distributions

Use `inspect_record(cell, repetition)` to retrieve one complete immutable record, including warnings and hashes. Runtime distributions retain all 50 datasets per cell; the logistic cell includes its bounded approximation outcomes.

In [ ]:
def inspect_record(cell: str, repetition: int = 0) -> dict:
    if cell not in cell_order:
        raise ValueError(f"Unknown cell {cell!r}; choose from {cell_order}")
    records = [record for record in result["records"] if record["cell"] == cell]
    if not 0 <= repetition < len(records):
        raise IndexError(f"repetition must be in [0, {len(records) - 1}]")
    return records[repetition]

example_record = inspect_record("prior-conflict-gaussian", 0)
print(json.dumps({key: example_record[key] for key in ["cell", "seed", "status", "true_boundaries", "predicted_boundaries", "k_map", "posterior_mass_at_k_max", "data_hash", "truth_hash", "effective_config_hash"]}, indent=2))

runtime_groups = [top_status_frame.loc[top_status_frame.cell == cell, "wall_seconds"].to_numpy() for cell in cell_order]
fig, ax = plt.subplots(figsize=(12, 4.5))
parts = ax.violinplot(runtime_groups, showmeans=True, showextrema=False)
for body in parts["bodies"]:
    body.set_facecolor("#00798C")
    body.set_alpha(0.55)
parts["cmeans"].set_color("#D1495B")
ax.set(title="Per-dataset runtime by cell", ylabel="seconds", xticks=np.arange(1, len(cell_order) + 1), xticklabels=cell_order)
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "runtime_distributions.png", bbox_inches="tight")
plt.show()
print(json.dumps(result["resources"], indent=2))

## 7. Generate an inspection report

The report exports flattened tables used by the plots and a compact JSON identity/quality summary. It deliberately labels the full result `pending-independent-review`; registering or interpreting the scientific result is a separate workflow.

In [ ]:
identity_frame.to_csv(OUTPUT_DIR / "identity_checks.csv", index=False)
seed_frame.to_csv(OUTPUT_DIR / "seed_audit.csv", index=False)
hash_audit.to_csv(OUTPUT_DIR / "hash_audit.csv", index=False)
top_status_frame.to_csv(OUTPUT_DIR / "top_level_statuses.csv", index=False)
metric_frame.to_csv(OUTPUT_DIR / "cell_metrics.csv", index=False)
paired_frame.to_csv(OUTPUT_DIR / "shared_paired_records.csv", index=False)
approximation_frame.to_csv(OUTPUT_DIR / "logistic_method_records.csv", index=False)
inspection_report = {
    "result_id": result["result_id"],
    "protocol_id": result["protocol_id"],
    "artifact_sha256": artifact_sha256,
    "code_commit": result["code"]["commit"],
    "plan_sha256": result["plan_sha256"],
    "environment_sha256": result["environment"]["sha256"],
    "record_count": len(result["records"]),
    "identity_checks_passed": bool(identity_frame.passed.all()),
    "seed_schedule_passed": bool((seed_frame.observed_seed == seed_frame.expected_seed).all()),
    "input_hashes_complete": bool(hash_audit[hash_names].all().all()),
    "top_level_status_counts": {key: int(value) for key, value in top_status_frame.status.value_counts().items()},
    "nested_method_status_counts": {f"{method}:{status}": int(count) for (method, status), count in method_status_frame.groupby(["method", "status"]).size().items()},
    "elapsed_wall_seconds": result["resources"]["elapsed_wall_seconds"],
    "peak_rss": result["resources"]["peak_rss"],
    "scientific_interpretation": result["scientific_interpretation"],
}
inspection_report["all_required_checks_passed"] = all([inspection_report["identity_checks_passed"], inspection_report["seed_schedule_passed"], inspection_report["input_hashes_complete"]])
(OUTPUT_DIR / "inspection_report.json").write_text(json.dumps(inspection_report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(inspection_report, indent=2))
assert result["scientific_interpretation"] == "pending-independent-review"
assert inspection_report["all_required_checks_passed"]